# 02 · Data frames & tibbles — importing the microbiome data

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}
cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

## Learning objectives

- Understand a **data frame / tibble**: a table of columns (variables) and rows (observations).
- **Import** CSV data with `readr::read_csv()` and know why tibbles are friendly.
- **Inspect** a dataset: `glimpse()`, `head()`, `summary()`, dimensions, column types.
- Pull out single columns and turn text columns into **factors**.
- Meet the three tables of our real dataset and how they connect.

---

## 0 · Setup — load packages and find the data

Every lesson starts with this small chunk. It loads the packages we need and
locates the `data/` folder whether you knit from the `notebooks/` folder or from
the project root.

In [ ]:
# library() loads an installed package so its functions become available.
library(readr)     # fast, friendly CSV import (read_csv)
library(dplyr)     # data wrangling verbs (a little here, a lot in lesson 03)
library(tibble)    # modern data frames ("tibbles")

# The data may sit at ../data (if knitting from notebooks/) or data/ (project root).
# This one-liner picks whichever path actually exists, so the lesson runs either way.
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"

> **Instructor note.** If someone gets a "file not found" error, it is almost
> always the working directory. Opening the `.Rproj` fixes this by anchoring the
> project root. The line above makes the lessons robust either way.

---

## 1 · The dataset we will use all session

We use the **HITChip Atlas** — a *real, published* human-gut microbiome study of
**1,006 healthy adults** (Lahti *et al.* 2014, *Nature Communications*). It comes
as three linked tables, the classic shape of omics data:

| Table | File | One row is… | Key columns |
|-------|------|-------------|-------------|
| **Sample metadata** | `sample_metadata.csv` | one participant/sample | `sample_id`, `age`, `sex`, `nationality`, `bmi_group`, `diversity_shannon` |
| **Taxonomy** | `taxonomy.csv` | one bacterial genus | `genus`, `phylum`, `family` |
| **Abundance** | `genus_abundance.csv` | one sample × 130 genera | `sample_id` + 130 abundance columns |

`sample_id` links metadata ↔ abundance; `genus` links taxonomy ↔ abundance
columns. We join them in lesson 03.

---

## 2 · Importing with `read_csv()`

`readr::read_csv()` reads a comma-separated file into a **tibble** (a modern data
frame). It prints a short message telling you the **type it guessed for each
column** — always read that message.

In [ ]:
# file.path() joins folder + file with the right separator ("../data/sample_metadata.csv").
# read_csv() reads that CSV into a tibble; watch the message reporting each column's type.
meta <- read_csv(file.path(data_dir, "sample_metadata.csv"))
meta                              # print the tibble: first 10 rows + dimensions + col types

Notice what a tibble does for you: it prints only the first 10 rows, shows the
**dimensions** (`1,006 × 7`) and the **type under each column name**
(`<chr>` = text, `<dbl>` = number). That saves you from flooding the console.

> Base R has `read.csv()` too; `read_csv()` (with the underscore) is faster,
> never silently turns text into factors, and gives that helpful type report.

---

## 3 · Inspecting a dataset

Before analysing anything, *look* at it. These are the commands you will run
first on every new table.

In [ ]:
dim(meta)        # the shape as c(rows, columns) = c(samples, variables)
nrow(meta)       # number of rows    -> here, participants/samples (1006)
ncol(meta)       # number of columns -> here, variables measured per sample
names(meta)      # the column names, i.e. the variable names in this table

`glimpse()` is the best single overview: it lists every column, its type, and
the first few values — transposed so wide tables still fit.

In [ ]:
glimpse(meta)    # one row per column: name, type (<chr>/<dbl>/…), and first values

`head()` / `tail()` show the first / last rows; `summary()` gives per-column
statistics (and, crucially, counts of `NA`).

In [ ]:
head(meta, 4)    # the first 4 rows — a quick sanity check that import worked
summary(meta)    # per-column stats: min/median/max for numbers, counts of NA's

> **Instructor note.** Point at the `summary()` output for `age`: it lists
> `NA's`. Real data is messy, and this dataset genuinely has missing ages and
> sexes, which we will handle honestly rather than hide.

### Exercise 1

Import `taxonomy.csv` into an object `taxonomy` and `genus_abundance.csv` into
`abund`. For each, print its **dimensions** and run `glimpse()` (for `abund`,
`glimpse()` will be long — that is fine).

<details><summary><b>Solution</b> (click to expand)</summary>

In [ ]:
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))

dim(taxonomy)   # 130 x 3
dim(abund)      # 1006 x 131 (sample_id + 130 genera)
glimpse(taxonomy)

</details>

In [ ]:
# (Loaded here so later chunks work even if the exercise was skipped.)
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))

---

## 4 · Getting at columns

A column is just a vector. Pull one out with `$` (or `[[ ]]`), then use anything
you learned in lesson 01.

In [ ]:
# 'table$column' pulls one column out as a plain vector — then use lesson-01 tools.
mean(meta$age, na.rm = TRUE)     # mean age; na.rm = TRUE skips participants with no age
range(meta$diversity_shannon)    # min and max Shannon diversity across all samples
length(unique(meta$nationality)) # unique() drops duplicates -> count of distinct regions

`table()` counts the categories in a text column — your first look at group sizes.

In [ ]:
table(meta$nationality)          # count samples per geographic region
table(meta$bmi_group)            # count samples per BMI class
table(meta$sex, useNA = "ifany") # useNA = "ifany" adds a column counting missing sex

---

## 5 · Turning text into factors

The category columns (`sex`, `nationality`, `bmi_group`) are text right now. For
grouping, ordered tables and plots, we convert them to **factors** and set a
meaningful order for BMI (lean → morbidly obese, not alphabetical).

In [ ]:
# Define the clinical order for BMI once, low -> high, so it isn't alphabetical.
bmi_levels <- c("underweight", "lean", "overweight",
                "obese", "severeobese", "morbidobese")

# '|>' pipes meta into mutate(); read it as "take meta, then change these columns".
meta <- meta |>
  mutate(
    sex         = factor(sex),          # text -> factor (categories from the data)
    nationality = factor(nationality),  # text -> factor, ditto
    bmi_group   = factor(bmi_group, levels = bmi_levels)  # + impose our chosen order
  )

# Confirm the factor now carries our order, not R's default alphabetical one.
levels(meta$bmi_group)     # underweight -> morbidobese, as we set
table(meta$bmi_group)      # the counts now print in that clinical order

`mutate()` (from **dplyr**) adds or changes columns — we meet it properly next
lesson; here it just lets us overwrite three columns cleanly.

> **Instructor note.** Ordering a factor once, here, pays off repeatedly: every
> later table and bar chart will list BMI groups in clinical order automatically.

### Exercise 2

(a) How many participants are in each `bmi_group` **after** ordering?
(b) What is the **median** `diversity_shannon` across all samples?
(c) Which `nationality` is the **most common** in the study?

<details><summary><b>Solution</b> (click to expand)</summary>

In [ ]:
table(meta$bmi_group)                       # (a)
median(meta$diversity_shannon)              # (b)
sort(table(meta$nationality), decreasing = TRUE)[1]  # (c) CentralEurope

</details>

---

## 6 · A first peek at the abundance table

The abundance table is **wide**: one row per sample, one column per genus. The
HITChip measures *genus-like groups*, so many column names carry an "et rel."
("and related") suffix, e.g. `Escherichia coli et rel.`. Let's confirm its shape
and look at three of those groups for the first few samples.

In [ ]:
dim(abund)                              # 1006 samples (rows) x 131 columns (id + 130 genera)
# table[rows, columns]: pick rows 1-5 and just four named columns to keep it readable.
abund[1:5, c("sample_id", "Akkermansia", "Escherichia coli et rel.",
             "Bacteroides fragilis et rel.")]

Because column names contain characters like spaces, we wrap them in backticks or
quote them when selecting — something we automate with **tidyr** in lesson 03 by
reshaping this wide table into a tidy long one.

---

## Recap

- A **tibble** is a friendly data frame: prints tidily, reports column types.
- Import with `read_csv()`; **read the column-type message** it prints.
- Inspect *before* analysing: `dim()`, `glimpse()`, `head()`, `summary()`.
- A column is a vector — reach it with `$`; count categories with `table()`.
- Convert category text to **factors** and set a meaningful **level order**.
- Our data is three linked tables (metadata, taxonomy, abundance) joined by
  `sample_id` and `genus`.

**Next:** lesson **03** — the wrangling core: `filter`, `select`, `mutate`,
`arrange`, `group_by`/`summarise`, **joins**, and `pivot_longer`/`pivot_wider`.